# Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [6]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [7]:
import os
import json
from typing import List
from pydantic import BaseModel, Field

import chromadb
from chromadb.utils import embedding_functions
from tavily import TavilyClient
from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [8]:
# Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY", OPENAI_API_KEY)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### ChromaDB — Load Existing Collection

In [9]:
# Reconnect to the persistent ChromaDB we built in Part 01.
# We must use the SAME embedding function used during ingestion.
chroma_client = chromadb.PersistentClient(path="chromadb")

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=CHROMA_OPENAI_API_KEY,
    model_name="text-embedding-ada-002"
)

collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

print(f"Collection loaded. Documents: {collection.count()}")

Collection loaded. Documents: 15


### Tools

Build at least 3 tools:
- `retrieve_game`: Semantic search against the vector DB
- `evaluate_retrieval`: LLM-as-judge to assess whether retrieved docs answer the question
- `game_web_search`: Tavily web search fallback when the vector DB is insufficient

#### Tool 1 — `retrieve_game`

In [10]:
@tool
def retrieve_game(query: str) -> str:
    """Semantic search: Finds the most relevant results in the vector DB.

    args:
    - query: a question about the game industry.

    You'll receive results as a list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = collection.query(
        query_texts=[query],
        n_results=5,
        include=["documents", "metadatas", "distances"]
    )

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    if not documents:
        return json.dumps({"results": [], "message": "No results found."})

    formatted = []
    for doc, meta, dist in zip(documents, metadatas, distances):
        formatted.append({
            "Name": meta.get("Name", "Unknown"),
            "Platform": meta.get("Platform", "Unknown"),
            "YearOfRelease": meta.get("YearOfRelease", "Unknown"),
            "Genre": meta.get("Genre", "Unknown"),
            "Publisher": meta.get("Publisher", "Unknown"),
            "Description": meta.get("Description", doc),
            "similarity_score": round(1 - dist, 4),
        })

    return json.dumps({"results": formatted})

# Quick smoke test
print(retrieve_game("Pokemon Gold Silver release year"))

{"results": [{"Name": "Pok\u00e9mon Gold and Silver", "Platform": "Game Boy Color", "YearOfRelease": 1999, "Genre": "Role-playing", "Publisher": "Nintendo", "Description": "Second-generation Pok\u00e9mon games introducing new regions, Pok\u00e9mon, and gameplay mechanics.", "similarity_score": 0.8566}, {"Name": "Pok\u00e9mon Ruby and Sapphire", "Platform": "Game Boy Advance", "YearOfRelease": 2002, "Genre": "Role-playing", "Publisher": "Nintendo", "Description": "Third-generation Pok\u00e9mon games set in the Hoenn region, featuring new Pok\u00e9mon and double battles.", "similarity_score": 0.8227}, {"Name": "Super Mario 64", "Platform": "Nintendo 64", "YearOfRelease": 1996, "Genre": "Platformer", "Publisher": "Nintendo", "Description": "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", "similarity_score": 0.7622}, {"Name": "Super Mario World", "Platform": "Super Nintendo Entertainment System (SNES)", "YearOfRelease

#### Tool 2 — `evaluate_retrieval`

In [11]:
# Pydantic model used by the judge LLM to produce structured output
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents are sufficient to answer the question")
    description: str = Field(description="Detailed explanation of why the documents are or are not useful")


@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> str:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    args:
    - question: original question from the user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    """
    judge_llm = LLM(model="gpt-4o-mini", temperature=0.0)

    judge_prompt = f"""Your task is to evaluate if the documents are enough to respond to the query.
Give a detailed explanation, so it's possible to take an action to accept it or not.

User question: {question}

Retrieved documents:
{retrieved_docs}

Evaluate:
1. Does any document directly address the question?
2. Is the information specific enough (e.g., correct platform, year, game name)?
3. Would a user be satisfied with this information as an answer?

Respond ONLY with a valid JSON object matching this schema:
{{"useful": <true|false>, "description": "<your explanation>"}}
"""

    response = judge_llm.invoke(judge_prompt)

    try:
        # Strip markdown fences if present
        content = response.content.strip()
        if content.startswith("```"):
            content = content.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
        report = EvaluationReport.model_validate_json(content)
    except Exception:
        # Fallback: check if retrieved_docs is non-empty and contains relevant info
        has_content = bool(retrieved_docs and retrieved_docs != '{"results": []}')
        report = EvaluationReport(
            useful=has_content,
            description="Evaluation parsing failed; falling back to content presence check."
        )

    return json.dumps(report.model_dump())


# Quick smoke test
sample_docs = retrieve_game("Pokemon Gold Silver release year")
print(evaluate_retrieval("When was Pokemon Gold and Silver released?", sample_docs))

{"useful": true, "description": "The first document retrieved directly addresses the user's question by providing the name of the game 'Pok\u00e9mon Gold and Silver', the platform 'Game Boy Color', and the year of release '1999'. This information is specific and relevant to the query, making it sufficient to answer the user's question. A user would likely be satisfied with this information as it directly answers when Pok\u00e9mon Gold and Silver was released."}


#### Tool 3 — `game_web_search`

In [12]:
@tool
def game_web_search(question: str) -> str:
    """Web search: Finds the most relevant results on the internet when the vector DB is insufficient.

    args:
    - question: a question about the game industry.
    """
    tavily = TavilyClient(api_key=TAVILY_API_KEY)

    response = tavily.search(
        query=question,
        search_depth="basic",
        max_results=5,
        include_answer=True,        # Ask Tavily for a direct summary answer
        include_raw_content=False,
    )

    # Extract the Tavily-generated answer + top result snippets
    answer = response.get("answer", "No direct answer available.")
    results = response.get("results", [])

    formatted_results = []
    for r in results:
        formatted_results.append({
            "title": r.get("title", ""),
            "url": r.get("url", ""),
            "content": r.get("content", "")[:400],  # Keep snippets concise
            "score": round(r.get("score", 0), 4),
        })

    output = {
        "tavily_answer": answer,
        "web_results": formatted_results,
    }

    return json.dumps(output)


# Quick smoke test
print(game_web_search("Was Mortal Kombat X released for PlayStation 5?"))

{"tavily_answer": "Mortal Kombat X was originally released for PlayStation 4, not PlayStation 5. It can be played on PS5 with updates.", "web_results": [{"title": "10 years ago today, Mortal Kombat X was released", "url": "https://www.facebook.com/gamebaseworld/posts/10-years-ago-today-mortal-kombat-x-was-released/997663085847715/", "content": "Mortal Kombat X released Tuesday for Xbox One, PlayStation 4, and PC. If anyone is a fan of fighting games at all, then check this one out", "score": 0.9997}, {"title": "Mortal Kombat X - PS5 Gameplay", "url": "https://www.youtube.com/watch?v=tqsw711ZuAk", "content": "Mortal Kombat X - PS5 Gameplay About this game : Mortal Kombat X is a 2015 fighting game developed by NetherRealm Studios and published by", "score": 0.9993}, {"title": "Mortal Kombat X - Wikipedia", "url": "https://en.wikipedia.org/wiki/Mortal_Kombat_X", "content": "***Mortal Kombat X*** is a 2015 fighting game developed by NetherRealm Studios and published by Warner Bros. An upgr

### Agent

Assemble the UdaPlay agent with:
- `gpt-4o-mini` as the reasoning backbone
- A system prompt that enforces the retrieve → evaluate → fallback workflow
- All three tools wired in

In [13]:
UDAPLAY_INSTRUCTIONS = """\
You are UdaPlay, an expert AI Research Agent specialising in the video game industry.

## Workflow — follow this order every time:

1. **Retrieve** — Always start by calling `retrieve_game` with the user's question
   to search the internal vector database for relevant game information.

2. **Evaluate** — Call `evaluate_retrieval` with the original question AND the documents
   returned in step 1. This tells you whether the retrieved docs are sufficient.

3. **Web search (if needed)** — If `evaluate_retrieval` returns `useful: false`,
   call `game_web_search` to find current information on the internet.

4. **Answer** — Synthesise all gathered information into a clear, concise, and
   accurate response. Always cite the source (internal DB or web).

## Rules:
- Never answer from memory alone — always retrieve first.
- Be specific: include game names, platforms, release years, and publishers when available.
- If information is genuinely unavailable even after a web search, say so honestly.
- Keep answers focused and factual. Avoid speculation.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=UDAPLAY_INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,   # Deterministic — important for factual research tasks
)

print("UdaPlay agent ready!")

UdaPlay agent ready!


### Invoke the Agent

In [14]:
def print_run(run, question: str):
    """Helper to display the agent's final answer and basic run stats."""
    final_state = run.get_final_state()
    messages = final_state.get("messages", [])

    # Find the last AI message with actual text content
    answer = "(no answer)"
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.content:
            answer = msg.content
            break

    # Count tool calls made during this run
    tool_calls_made = []
    for msg in messages:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            tool_calls_made.extend([tc.function.name for tc in msg.tool_calls])

    total_tokens = final_state.get("total_tokens", 0)

    print(f"❓ Question : {question}")
    print(f"🔧 Tools used: {tool_calls_made}")
    print(f"🪙 Tokens   : {total_tokens}")
    print(f"\n💬 Answer:\n{answer}")
    print("\n" + "=" * 70 + "\n")

In [15]:
# Q1 — Should be answered from the vector DB
q1 = "When were Pokémon Gold and Silver released?"
run1 = agent.invoke(q1, session_id="demo")
print_run(run1, q1)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
❓ Question : When were Pokémon Gold and Silver released?
🔧 Tools used: ['retrieve_game', 'evaluate_retrieval']
🪙 Tokens   : 3512

💬 Answer:
Pokémon Gold and Silver were released for the Game Boy Color in 1999. These games are part of the second generation of Pokémon and introduced new regions, Pokémon, and gameplay mechanics (Nintendo).




In [16]:
# Q2 — Should be answered from the vector DB
q2 = "Which one was the first 3D platformer Mario game?"
run2 = agent.invoke(q2, session_id="demo")
print_run(run2, q2)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
❓ Question : Which one was the first 3D platformer Mario game?
🔧 Tools used: ['retrieve_game', 'evaluate_retrieval', 'retrieve_game', 'evaluate_retrieval']
🪙 Tokens   : 6635

💬 Answer:
The first 3D platformer Mario game is **Super Mario 64**, which was released for the Nintendo 64 in 1996. This game is considered groundbreaking as it set new standards for the platforming genre, featuring Mario's quest to rescue Princess Peach (Nintendo).




In [17]:
# Q3 — May require web search (MK X is older; PS5 details may not be in the DB)
q3 = "Was Mortal Kombat X released for PlayStation 5?"
run3 = agent.invoke(q3, session_id="demo")
print_run(run3, q3)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
❓ Question : Was Mortal Kombat X released for PlayStation 5?
🔧 Tools used: ['retrieve_game', 'evaluate_retrieval', 'retrieve_game', 'evaluate_retrieval', 'retrieve_game', 'evaluate_retrieval', 'game_web_search']
🪙 Tokens   : 13966

💬 Answer:
Mortal Kombat X was originally released for the PlayStation 4 in 2015. While it was not specifically released for the PlayStation 5, it can be played on the PS5 with updates (source: various web results).




---
### (Optional) Advanced — Long-Term Memory

Extend the agent with a `LongTermMemory` store so it can remember facts across sessions.

In [18]:
from lib.memory import LongTermMemory, MemoryFragment, TimestampFilter
from lib.vector_db import VectorStoreManager

# Create an in-memory vector store manager for long-term memory
vsm = VectorStoreManager(openai_api_key=OPENAI_API_KEY)
ltm = LongTermMemory(db=vsm)

# --- Store a fact discovered during the session ---
ltm.register(
    MemoryFragment(
        content="Pokémon Gold and Silver were released in 1999 (Japan) and 2000 (NA/EU) for Game Boy Color.",
        owner="user_001",
        namespace="game_facts"
    )
)

ltm.register(
    MemoryFragment(
        content="Super Mario 64 (1996) for Nintendo 64 was the first 3D platformer in the Mario franchise.",
        owner="user_001",
        namespace="game_facts"
    )
)

print("Long-term memory fragments registered.")

Long-term memory fragments registered.


In [19]:
# --- Retrieve a previously stored fact ---
memory_result = ltm.search(
    query_text="3D Mario game Nintendo 64",
    owner="user_001",
    namespace="game_facts",
    limit=2
)

print("Recalled memories:")
for frag, dist in zip(memory_result.fragments, memory_result.metadata["distances"]):
    print(f"  [{frag.namespace}] (score {1 - dist:.4f}) {frag.content}")

Recalled memories:
  [game_facts] (score 0.9223) Super Mario 64 (1996) for Nintendo 64 was the first 3D platformer in the Mario franchise.
  [game_facts] (score 0.7930) Pokémon Gold and Silver were released in 1999 (Japan) and 2000 (NA/EU) for Game Boy Color.


### (Optional) Advanced — State Machine Agent with Pre-Defined Nodes

Convert the agent to an explicit state machine where each tool occupies a dedicated node.

In [20]:
from typing import TypedDict, Optional, List
from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Resource
from lib.llm import LLM
from lib.messages import SystemMessage, UserMessage, AIMessage


class UdaPlayState(TypedDict):
    question: str
    retrieved_docs: Optional[str]
    evaluation: Optional[str]
    web_results: Optional[str]
    final_answer: Optional[str]
    needs_web_search: bool


def retrieve_node(state: UdaPlayState) -> UdaPlayState:
    """Node: retrieve from vector DB."""
    docs = retrieve_game(state["question"])
    return {"retrieved_docs": docs}


def evaluate_node(state: UdaPlayState) -> UdaPlayState:
    """Node: judge whether retrieved docs are sufficient."""
    evaluation = evaluate_retrieval(state["question"], state["retrieved_docs"])
    eval_dict = json.loads(evaluation)
    return {
        "evaluation": evaluation,
        "needs_web_search": not eval_dict.get("useful", True),
    }


def web_search_node(state: UdaPlayState) -> UdaPlayState:
    """Node: search the web if evaluation says docs are insufficient."""
    web_results = game_web_search(state["question"])
    return {"web_results": web_results}


def answer_node(state: UdaPlayState, resource: Resource) -> UdaPlayState:
    """Node: synthesise all gathered info into a final answer."""
    llm: LLM = resource.vars["llm"]

    context_parts = []
    if state.get("retrieved_docs"):
        context_parts.append(f"Internal DB results:\n{state['retrieved_docs']}")
    if state.get("web_results"):
        context_parts.append(f"Web search results:\n{state['web_results']}")

    context = "\n\n".join(context_parts)

    prompt = (
        f"Answer the following question about video games using the provided context.\n"
        f"Be specific: include names, platforms, and release years where available.\n\n"
        f"Question: {state['question']}\n\n"
        f"Context:\n{context}\n\n"
        f"Answer:"
    )

    response = llm.invoke(prompt)
    return {"final_answer": response.content}


# --- Build the explicit state machine ---
sm = StateMachine[UdaPlayState](UdaPlayState)

entry       = EntryPoint[UdaPlayState]()
retrieve    = Step[UdaPlayState]("retrieve",    retrieve_node)
evaluate    = Step[UdaPlayState]("evaluate",    evaluate_node)
web_search  = Step[UdaPlayState]("web_search",  web_search_node)
answer      = Step[UdaPlayState]("answer",      answer_node)
termination = Termination[UdaPlayState]()

sm.add_steps([entry, retrieve, evaluate, web_search, answer, termination])

sm.connect(entry,    retrieve)
sm.connect(retrieve, evaluate)

# Conditional branch: go to web_search only when docs are insufficient
def route_after_evaluate(state: UdaPlayState):
    return web_search if state["needs_web_search"] else answer

sm.connect(evaluate,   [web_search, answer], route_after_evaluate)
sm.connect(web_search, answer)
sm.connect(answer,     termination)

resource = Resource(vars={"llm": LLM(model="gpt-4o-mini", temperature=0.0)})

print("Explicit state-machine agent built!")

Explicit state-machine agent built!


In [21]:
# Run the state-machine agent
questions = [
    "When were Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?",
]

for q in questions:
    initial: UdaPlayState = {
        "question": q,
        "retrieved_docs": None,
        "evaluation": None,
        "web_results": None,
        "final_answer": None,
        "needs_web_search": False,
    }
    run = sm.run(initial, resource=resource)
    final = run.get_final_state()

    eval_dict = json.loads(final.get("evaluation") or '{"useful": null}')
    used_web = final.get("needs_web_search", False)

    print(f"❓ {q}")
    print(f"   DB useful: {eval_dict.get('useful')} | Web used: {used_web}")
    print(f"   💬 {final['final_answer']}")
    print("=" * 70)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: evaluate
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__
❓ When were Pokémon Gold and Silver released?
   DB useful: True | Web used: False
   💬 Pokémon Gold and Silver were released for the Game Boy Color in 1999.
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: evaluate
[StateMachine] Executing step: answer
[StateMachine] Terminating: __termination__
❓ Which one was the first 3D platformer Mario game?
   DB useful: True | Web used: False
   💬 The first 3D platformer Mario game is **Super Mario 64**, released in **1996** for the **Nintendo 64**. This groundbreaking game set new standards for the genre, featuring Mario's quest to rescue Princess Peach.
[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: evaluate
[StateMachine] Executing st